# FlashAttention from Scratch (CUDA)
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**

The first compile takes 1–2 minutes. Results: a table + `results.png`.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## Kernel

In [ ]:
%%writefile flash.cu
// FlashAttention forward pass from scratch (CUDA, FP32)
//
// Computes O = softmax(Q K^T / sqrt(d)) V without ever building the N x N score matrix.
//   * Tiling:        K and V are processed in small tiles held in shared memory.
//   * Online softmax: a running max (m) and running sum (l) are updated tile by tile,
//                     and the partial output is rescaled, so the result is exact.
//   * Memory:        only the output O is allocated -> extra memory is O(N), not O(N^2).
//
// Layout: Q, K, V, O are contiguous float32 tensors of shape (B, H, N, D).
// Supported head sizes D: 32, 64, 128.  Optional causal mask.
//
// Mapping: one thread owns one query row (its q[] and o[] live in registers).
// A block of BR threads handles BR query rows and loops over all K/V tiles.

#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <cuda_runtime.h>
#include <cmath>

template <int D, int BR, int BC>
__global__ void __launch_bounds__(BR)
flash_fwd_kernel(const float* __restrict__ Q, const float* __restrict__ K,
                 const float* __restrict__ V, float* __restrict__ O,
                 int N, float scale, bool causal)
{
    __shared__ float Ks[BC][D];
    __shared__ float Vs[BC][D];

    const int    tid    = threadIdx.x;
    const int    qi     = blockIdx.x * BR + tid;          // query row owned by this thread
    const size_t base   = (size_t)blockIdx.y * N * D;     // start of this (batch, head)
    const bool   active = qi < N;

    float q[D], o[D];
    #pragma unroll
    for (int d = 0; d < D; d++) {
        o[d] = 0.f;
        q[d] = active ? Q[base + (size_t)qi * D + d] * scale : 0.f;   // fold 1/sqrt(D) into q
    }
    float m = -INFINITY;   // running row max
    float l = 0.f;         // running row sum of exp

    // With a causal mask, no key beyond the last query row of this block is ever needed.
    int kv_end = N;
    if (causal) kv_end = min(N, (int)((blockIdx.x + 1) * BR));

    for (int kv0 = 0; kv0 < kv_end; kv0 += BC) {
        __syncthreads();                                   // everyone finished with the old tile
        for (int idx = tid; idx < BC * D; idx += BR) {     // cooperative load of K and V tiles
            int r = idx / D, c = idx % D, row = kv0 + r;
            float kk = 0.f, vv = 0.f;
            if (row < N) { kk = K[base + (size_t)row * D + c]; vv = V[base + (size_t)row * D + c]; }
            Ks[r][c] = kk;
            Vs[r][c] = vv;
        }
        __syncthreads();                                   // tile is ready

        if (active) {
            // 1) scores of this query row against the BC keys in the tile
            float s[BC];
            float m_tile = -INFINITY;
            #pragma unroll
            for (int j = 0; j < BC; j++) {
                float acc = 0.f;
                #pragma unroll
                for (int d = 0; d < D; d++) acc += q[d] * Ks[j][d];
                int key = kv0 + j;
                bool masked = (key >= N) || (causal && key > qi);
                s[j] = masked ? -INFINITY : acc;
                m_tile = fmaxf(m_tile, s[j]);
            }

            // 2) online softmax update
            float m_new = fmaxf(m, m_tile);
            if (m_new > -INFINITY) {
                float alpha = expf(m - m_new);             // rescale what we had so far
                l *= alpha;
                #pragma unroll
                for (int d = 0; d < D; d++) o[d] *= alpha;
                #pragma unroll
                for (int j = 0; j < BC; j++) {
                    float p = expf(s[j] - m_new);          // masked entries give exactly 0
                    l += p;
                    #pragma unroll
                    for (int d = 0; d < D; d++) o[d] += p * Vs[j][d];
                }
                m = m_new;
            }
        }
    }

    if (active) {
        float inv = 1.f / l;
        #pragma unroll
        for (int d = 0; d < D; d++) O[base + (size_t)qi * D + d] = o[d] * inv;
    }
}

torch::Tensor flash_forward(torch::Tensor Q, torch::Tensor K, torch::Tensor V, bool causal)
{
    TORCH_CHECK(Q.is_cuda() && K.is_cuda() && V.is_cuda(), "Q, K, V must be CUDA tensors");
    TORCH_CHECK(Q.dtype() == torch::kFloat32 && K.dtype() == torch::kFloat32 &&
                V.dtype() == torch::kFloat32, "only float32 is supported");
    TORCH_CHECK(Q.dim() == 4, "expected shape (B, H, N, D)");
    TORCH_CHECK(Q.sizes() == K.sizes() && Q.sizes() == V.sizes(), "Q, K, V must have the same shape");
    Q = Q.contiguous(); K = K.contiguous(); V = V.contiguous();

    const int B = Q.size(0), H = Q.size(1), N = Q.size(2), D = Q.size(3);
    auto O = torch::empty_like(Q);
    const float scale = 1.0f / sqrtf((float)D);
    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

#define LAUNCH(D_, BR_, BC_)                                                           \
    {                                                                                  \
        dim3 grid((N + BR_ - 1) / BR_, B * H);                                         \
        flash_fwd_kernel<D_, BR_, BC_><<<grid, BR_, 0, stream>>>(                      \
            Q.data_ptr<float>(), K.data_ptr<float>(), V.data_ptr<float>(),             \
            O.data_ptr<float>(), N, scale, causal);                                    \
    }
    if      (D == 32)  LAUNCH(32, 128, 32)
    else if (D == 64)  LAUNCH(64, 128, 32)
    else if (D == 128) LAUNCH(128, 64, 16)
    else TORCH_CHECK(false, "head size D must be 32, 64 or 128");
#undef LAUNCH

    TORCH_CHECK(cudaGetLastError() == cudaSuccess, "flash_fwd_kernel launch failed");
    return O;
}


## PyTorch binding

In [ ]:
%%writefile main.cpp
#include <torch/extension.h>

torch::Tensor flash_forward(torch::Tensor Q, torch::Tensor K, torch::Tensor V, bool causal);

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &flash_forward, "FlashAttention forward (online softmax, tiled, FP32)",
          pybind11::arg("Q"), pybind11::arg("K"), pybind11::arg("V"), pybind11::arg("causal") = false);
}


## Benchmark script

In [ ]:
%%writefile benchmark.py
"""Benchmark: our CUDA FlashAttention forward vs PyTorch attention (time + extra memory)."""
import argparse, math, os, time
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load

p = argparse.ArgumentParser()
p.add_argument("--causal", action="store_true", help="use a causal mask")
p.add_argument("--batch", type=int, default=2)
p.add_argument("--heads", type=int, default=8)
p.add_argument("--dim", type=int, default=64, choices=[32, 64, 128])
p.add_argument("--sizes", type=int, nargs="+", default=[512, 1024, 2048, 4096, 8192])
p.add_argument("--iters", type=int, default=20)
args = p.parse_args()

here = os.path.dirname(os.path.abspath(__file__))
print("Compiling CUDA extension (first time takes ~1-2 minutes)...")
flash = load(name="flash_ext", sources=[os.path.join(here, "main.cpp"), os.path.join(here, "flash.cu")],
             extra_cuda_cflags=["-O3"], verbose=False)
dev = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print(f"B={args.batch} H={args.heads} D={args.dim} causal={args.causal} dtype=float32\n")


def naive_attention(q, k, v, causal):
    # the textbook version: builds the full N x N score matrix
    s = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if causal:
        n = q.shape[-2]
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=q.device), 1)
        s = s.masked_fill(mask, float("-inf"))
    return torch.softmax(s, dim=-1) @ v


def sdpa(q, k, v, causal):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal)


def ours(q, k, v, causal):
    return flash.forward(q, k, v, causal)


def measure(fn, q, k, v, causal):
    """returns (ms per call, extra peak memory in MB) or (None, None) if out of memory"""
    try:
        for _ in range(2):
            fn(q, k, v, causal)
        torch.cuda.synchronize(); torch.cuda.empty_cache()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
        fn(q, k, v, causal); torch.cuda.synchronize()
        peak_mb = (torch.cuda.max_memory_allocated() - base) / 2**20
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(args.iters):
            fn(q, k, v, causal)
        e.record(); torch.cuda.synchronize()
        return s.elapsed_time(e) / args.iters, peak_mb
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None


def fmt(x, spec):
    return "   OOM" if x is None else format(x, spec)


rows = []
hdr = f"{'N':>6} | {'naive ms':>9} {'SDPA ms':>9} {'ours ms':>9} | {'naive MB':>9} {'SDPA MB':>9} {'ours MB':>9} | {'max err':>9}"
print(hdr); print("-" * len(hdr))
torch.manual_seed(0)
for n in args.sizes:
    q, k, v = [torch.randn(args.batch, args.heads, n, args.dim, device=dev) for _ in range(3)]
    t_n, m_n = measure(naive_attention, q, k, v, args.causal)
    t_s, m_s = measure(sdpa, q, k, v, args.causal)
    t_o, m_o = measure(ours, q, k, v, args.causal)
    # correctness: compare to SDPA (always fits) in float32
    err = (ours(q, k, v, args.causal) - sdpa(q, k, v, args.causal)).abs().max().item()
    print(f"{n:>6} | {fmt(t_n,'9.3f')} {fmt(t_s,'9.3f')} {fmt(t_o,'9.3f')} | "
          f"{fmt(m_n,'9.1f')} {fmt(m_s,'9.1f')} {fmt(m_o,'9.1f')} | {err:9.2e} {'OK' if err < 1e-3 else 'MISMATCH'}")
    rows.append((n, t_n, t_s, t_o, m_n, m_s, m_o))
    del q, k, v; torch.cuda.empty_cache()

# chart
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    xs = [r[0] for r in rows]
    for idx, name in [(1, "naive PyTorch"), (2, "PyTorch SDPA"), (3, "ours (CUDA)")]:
        pts = [(r[0], r[idx]) for r in rows if r[idx] is not None]
        ax[0].plot(*zip(*pts), marker="o", label=name)
    for idx, name in [(4, "naive PyTorch"), (5, "PyTorch SDPA"), (6, "ours (CUDA)")]:
        pts = [(r[0], r[idx]) for r in rows if r[idx] is not None]
        ax[1].plot(*zip(*pts), marker="o", label=name)
    ax[0].set_title("Time per call (ms)"); ax[1].set_title("Extra GPU memory (MB)")
    for a in ax:
        a.set_xscale("log", base=2); a.set_yscale("log"); a.set_xlabel("sequence length N"); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig("results.png", dpi=150)
    print("\nSaved chart: results.png")
except Exception as ex:
    print("chart skipped:", ex)


## Run: no mask

In [ ]:
!python benchmark.py

## Run: causal mask

In [ ]:
!python benchmark.py --causal

## Chart

In [ ]:
from IPython.display import Image
Image('results.png')